# ReMask — thí nghiệm bổ sung cho rebuttal

**Luồng dùng ZIP:** `git clone https://github.com/Kuan2vn/ReMask.git` → `%cd /content/ReMask` → giải nén `remask_datasets.zip` **ngay tại ReMask**. ZIP đã có tiền tố `data/`, nên kết quả phải là `ReMask/data/SUIM/...`, không phải `ReMask/data/data/SUIM`. Notebook dùng được cả thư mục clone thủ công và thư mục nằm dưới `SND_ReMask/ReMask`; repository GitHub hiện tên là **ReMask**.

Chạy các cell thiết lập, rồi các nhóm thí nghiệm theo thứ tự ưu tiên bên dưới. Mỗi nhóm có cell riêng; không cần chờ hết ma trận mới xuất báo cáo. Chọn GPU và High-RAM; A100 phù hợp hơn cho recipe gốc batch4, FP32. Smoke nhỏ chỉ kiểm tra tính đúng, chưa xác nhận GPU đủ bộ nhớ ở full resolution.

Kết quả và phiên bản code lưu trên Drive. Nhiều phiên Colab là cần thiết; chạy lại cùng notebook/output để tiếp tục checkpoint đã commit. Các kỳ vọng bên dưới là **cách đọc kết quả**, không phải số liệu dự đoán hay điều kiện lọc kết quả có lợi.


## 1. Đường dẫn và phiên bản code

Nếu đã clone, `%cd` đến root `ReMask` trước cell này hoặc đặt `REPO_HINT`. `DATA_ZIP` trỏ đến ZIP trên Drive; để trống nếu đã tự giải nén. Không cần HF_TOKEN khi dùng dữ liệu sẵn có. Chỉ cần token có quyền đọc SUIM private nếu chọn tải dự phòng từ Hugging Face.

Mặc định hai protocol, seed1, panel cố định 128 ảnh train **không dùng mask**, đo mỗi 500 updates. Không đổi panel/seed/recipe giữa các nhánh so sánh. Output v3 chứa bản sửa resume CUDA. Giữ nguyên output/code pin v1/v2 cũ; không trộn các phiên bản code trong một output.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time, venv
from google.colab import drive

SESSION_STARTED = time.monotonic()
SESSION_HOURS = 9.0
OUTPUT = Path('/content/drive/MyDrive/ReMask_component_experiments_v3')
REPO_HINT = None  # ví dụ '/content/SND_ReMask/ReMask'
DATA_ZIP = Path('/content/drive/MyDrive/remask_datasets.zip')  # hoặc None nếu đã giải nén
PROTOCOL = 'all'  # uiis_to_suim / usis10k_to_suim / all
SEED = 1
PANEL_IMAGES = 128
DO_FROZEN = False  # phân tích source-only tùy chọn; ưu tiên F/mismatch trước
RUN_OPTIONAL = False  # nhóm P2 cuối notebook; tất cả vẫn có sẵn

drive.mount('/content/drive')
candidates = ([Path(REPO_HINT)] if REPO_HINT else []) + [Path.cwd(), Path.cwd()/'ReMask',
    Path('/content/ReMask'), Path('/content/SND_ReMask/ReMask'), Path('/content/SND_Remask/ReMask')]
REPO = next((p.resolve() for p in candidates if (p/'experiments/rebuttal/run.py').is_file() and (p/'.git').exists()), None)
if REPO is None:
    if REPO_HINT:
        raise ValueError('REPO_HINT không trỏ tới checkout ReMask')
    REPO = Path('/content/ReMask')
    subprocess.check_call(['git', 'clone', 'https://github.com/Kuan2vn/ReMask.git', str(REPO)])
os.chdir(REPO)
sys.path.insert(0, str(REPO/'experiments/rebuttal'))
from colab_setup import pin_revision
commit = pin_revision(REPO, OUTPUT)
print('Repository:', REPO, '| code revision:', commit)
DATA = REPO/'data/SUIM'


## 2. Runtime và dữ liệu — chạy một lần mỗi phiên

Cell cài đúng dependencies trong môi trường riêng để tránh xung đột gói có sẵn trên Colab. Với ZIP: xác minh SHA256 và cấu trúc 44.432 file, giải nén vào root repo. Có receipt thì không giải nén lại. ZIP khoảng 1,26 GB; sau giải nén khoảng 7,89 GB, chưa tính checkpoint/output. Không tải lại SUIM nếu dữ liệu đã có.

ZIP có SUIM 1.525/110, UIIS 3.937/691 và USIS10K 7.442/1.594/1.596 ảnh cùng semantic masks. UIIS dùng RGB/BMP palette UIIS; USIS10K có cả background và ignore. Thí nghiệm SFUDA dưới đây chỉ cần ảnh train SUIM, mask TEST và source checkpoints; UIIS/USIS10K trong ZIP phục vụ kiểm tra/tái huấn luyện source riêng. Source checkpoints tự tải và kiểm SHA256, không nằm trong ZIP. Không sửa/re-encode SUIM gốc.


In [ ]:
ENV = Path('/content/remask-runtime')
PYTHON = ENV/'bin/python'
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(ENV)
subprocess.check_call([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install', '--upgrade', 'pip'])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'install', 'torch==2.8.0', 'torchvision==0.23.0', '--index-url', 'https://download.pytorch.org/whl/cu126'])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'install', '-r', str(REPO/'requirements.txt')])
subprocess.check_call([str(PYTHON), '-m', 'pip', 'check'])
subprocess.check_call(['nvidia-smi'])

def run_script(script, *args, env=None):
    subprocess.check_call([str(PYTHON), str(REPO/script), *map(str, args)], cwd=REPO, env=env)

archives = ([Path(DATA_ZIP)] if DATA_ZIP is not None else []) + [REPO/'remask_datasets.zip', Path('/content/remask_datasets.zip')]
archive = next((p for p in archives if p.is_file()), None)
if archive:
    run_script('experiments/rebuttal/bundle.py', '--archive', archive, '--repo', REPO)
elif not DATA.exists():
    from google.colab import userdata
    token = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
    prepare = r"""
from pathlib import Path
import hashlib, sys, zipfile
from huggingface_hub import hf_hub_download
sys.path.insert(0, 'experiments/rebuttal')
from bundle import safe_members, extract
archive = hf_hub_download('kuan2/SUIM-ReMask', 'SUIM.zip', repo_type='dataset', revision='b2997099ded641a1de08d00072bfb268cb9ef6dc')
if hashlib.sha256(Path(archive).read_bytes()).hexdigest() != '86943900de1a74feaf826e0f2e8aa1e2067339c70a6755f91b8bb30522c874cd':
    raise RuntimeError('SUIM archive checksum mismatch')
with zipfile.ZipFile(archive) as bundle:
    extract(bundle, Path.cwd()/'data', safe_members(bundle, prefix='SUIM'))
"""
    subprocess.check_call([str(PYTHON), '-c', prepare], cwd=REPO, env=dict(os.environ, HF_TOKEN=token))
    del token
else:
    print('Dùng SUIM đã giải nén; không tải lại. Không có ZIP để kiểm SHA256 tại phiên này.')
from bundle import check_suim
check_suim(DATA)
run_script('experiments/rebuttal/run.py', '--check', '--protocol', PROTOCOL, '--data', DATA)


## 3. Kiểm tra chạy và khôi phục checkpoint

**Mục đích:** kiểm tra model, loss, data view, pseudo-label refresh, map chẩn đoán và khôi phục trong process mới trước khi tốn nhiều GPU, gồm kiểm tra riêng buffer prototype từ CPU checkpoint lên CUDA. Chỉ 8 ảnh train/2 TEST, 2+2 updates và input nhỏ; không dùng các con số smoke trong rebuttal. Kết quả đúng là các phase có `complete.json`, checkpoint/bank được kiểm hash, không phải một mIoU mục tiêu.


In [ ]:
run_script('experiments/rebuttal/tests/test_cuda_resume.py', '-v')
COMMON = ['--protocol', PROTOCOL, '--data', DATA, '--output', OUTPUT, '--seed', SEED,
          '--panel-images', PANEL_IMAGES, '--save-pixel-diagnostics']
run_script('experiments/rebuttal/run.py', '--arm', 'F', *COMMON, '--smoke', '--smoke-refresh')

def remaining_hours():
    return SESSION_HOURS - (time.monotonic()-SESSION_STARTED)/3600

def run_arms(*arms):
    for arm in arms:
        hours = remaining_hours()
        if hours <= 0.05:
            print('Hết ngân sách phiên; giữ output và chạy lại notebook để tiếp tục:', arm)
            return
        run_script('experiments/rebuttal/run.py', '--arm', arm, *COMMON, '--session-hours', hours)

def export_reports():
    run_script('experiments/rebuttal/report.py', '--output', OUTPUT)
    run_script('experiments/rebuttal/dynamics.py', '--output', OUTPUT, '--arm', 'F')


## P0 — F và mismatch theo iteration: câu hỏi stage toàn cục hay theo pixel

**Reviewer RgDe:** ảnh/class/vùng có thể ổn định ở thời điểm khác nhau, tại sao mọi pixel chuyển UA→TC cùng lúc? **QNjz:** bằng chứng cho split 25k/5k và lợi ích của UA/TC theo thời gian.

**Chạy gì:** reference F gốc 25k UA + 5k TC, tổng 30k updates/protocol. Tại mỗi checkpoint 500 updates, teacher và student dự đoán **cùng view xác định** trên 128 ảnh train không nhãn đã chọn trước. Lưu mismatch tổng, foreground, class-macro, từng predicted class, từng ảnh và map stride8; panel giống nhau giữa các nhánh. Chẩn đoán không can thiệp loss, không gán Stage2 cho pixel và không dùng mask train.

**Cell báo cáo kế tiếp:** đồ thị theo iteration, CSV cho các cửa sổ 0–5k,5–10k,…,20–25k; thêm phân bố theo ảnh/class và tỷ lệ pixel vừa T=S ở cả 3 lần đo liên tiếp vừa có teacher label không đổi. Đây là proxy ổn định, không phải tỷ lệ pixel “đủ đúng để vào Stage2”. Không đặt trước ngưỡng 5%.

**Kỳ vọng/cách đọc:** nếu trung bình giảm nhưng class/vùng ổn định khác nhau, kết quả cho thấy curriculum toàn cục vẫn vận hành cùng trọng số reliability cục bộ; không cần khẳng định mọi pixel đã hội tụ ở 25k. Nếu đường cong chưa phẳng hoặc foreground khác nhiều, báo cáo đúng phân bố đó. Mismatch thấp không tự chứng minh pseudo-label đúng, 25k tối ưu hoặc adaptive pixel switching kém hơn. Đọc thêm score-quality trên TEST và các continuation controls bên dưới. T=S ở khởi tạo và ngay khi reset teacher không được dùng làm bằng chứng ổn định.

**Ngân sách:** 30k/protocol, reference được tái sử dụng cho các nhánh TC/stage. Đã có checkpoint một phần vẫn xuất được báo cáo; không cần chạy lại F chỉ để vẽ.


In [ ]:
run_arms('F')
export_reports()


## P0 — Xuất riêng báo cáo dynamics cho supervisor, không chạy training

Cell này có thể chạy lại bất cứ lúc nào sau setup. Chỉ đọc evaluation đã commit; dữ liệu chưa tới 20k/25k được ghi là chưa đo. Tách Stage A và B để tránh hiểu nhầm hiệu ứng teacher reset. Run cũ thiếu map vẫn có thống kê tổng; không tự bịa phân bố pixel.

File cần gửi: `full/dynamics/<protocol>/seed1/F/A/SUPERVISOR_REPORT.md`, `dynamics.png`, `over_time.csv`, `per_image.csv`, `pixel_stability_by_class.csv`; map minh họa ở đúng 20k/25k nếu đã đo. `full/mismatch_5k_windows.csv` là tổng hợp các snapshot trong mỗi cửa sổ, không phải đo mọi optimizer step. Smoke nằm riêng dưới `smoke/`.


In [ ]:
export_reports()
from IPython.display import display, Markdown, Image as DisplayImage
for report in sorted((OUTPUT/'full/dynamics').glob('*/seed*/F/*/SUPERVISOR_REPORT.md')):
    display(Markdown(report.read_text()))
    plot = report.parent/'dynamics.png'
    if plot.exists():
        display(DisplayImage(filename=str(plot)))
print('Báo cáo và CSV:', OUTPUT/'full')


## P1 — TC: tách temporal consistency và class balancing

**Reviewer Vc4u:** Table4 bật/tắt TC như một khối, chưa tách hai thành phần. Chạy `TC-neither`, `TC-Tonly`, `TC-Conly`; F/B là ô có cả hai. Cả bốn bắt đầu từ **cùng F/A@25k teacher**, cùng reset/LR/UWRG, 5k updates/nhánh. Chi phí bổ sung 15k/protocol.

**Đọc kết quả:** bảng 2×2 với mIoU, foreground, IoU từng class và effective CE class mass; báo cáo cả endpoint30k và peak tại cùng checkpoint cho mọi metric. Tương tác hoặc lợi ích riêng ở class hiếm vẫn có ý nghĩa; không kỳ vọng mỗi thành phần đều tăng tất cả class. Đây là đối chiếu thành phần có kiểm soát, không so các peak lấy từ những class khác nhau.


In [ ]:
run_arms('TC-neither', 'TC-Tonly', 'TC-Conly')
export_reports()


## P1 — UA→TC: đổi reliability hay chỉ thêm thời gian/reset?

**QNjz và RgDe:** mismatch bổ sung bằng chứng cơ chế nhưng không tách được tác động của tiếp tục UA, reset và đổi LR. Chạy `S-UA-restart` (tiếp UA sau cùng reset/LR như F/B) và `S-UA-continuous` (giữ toàn bộ student/teacher/optimizer UA từ25k đến30k). Mỗi nhánh thêm5k/protocol, không thay đổi tổng budget30k.

**Đọc kết quả:** F tốt hơn UA-restart là bằng chứng trực tiếp hơn cho thay reliability ở late stage; UA-continuous kiểm tra “chỉ train UA lâu hơn”. Nếu không hơn, thu hẹp kết luận đúng điều đo được. Các control này không phải phương pháp chuyển stage riêng cho từng pixel. Không dùng mức mismatch sau reset như bằng chứng 25k hội tụ.


In [ ]:
run_arms('S-UA-restart', 'S-UA-continuous')
export_reports()


## P1 — UWRG so với ordinary augmentation: thay probe có kiểm soát

**RgDe, Vc4u; liên quan underwater motivation của5EXK:** underwater perturbation có hơn generic augmentation hay chỉ do thêm view/regularization? `G-generic-med` thay underwater probe bằng ColorJitter trong cả hai stage; giữ số forward, aggregation median, floor/lambda, schedule và training augmentation gốc. Độ thay đổi RGB được match trên ảnh train không nhãn bằng quy tắc cố định.

**Ngân sách:** thêm30k/protocol. **Đọc kết quả:** mIoU/foreground/boundary và cost trên cùng budget; generic đang thay probe tạo reliability, không thêm một loss augmentation khác. RGB-magnitude matching không có nghĩa hai perturbation tương đương về ngữ nghĩa. Nếu UW có lợi ở một số điều kiện thì dùng toàn bộ strata đã khai báo, không chọn riêng strata theo gain. Frozen probe bên dưới là bổ trợ, không thay thế run adaptation này.


In [ ]:
run_arms('G-generic-med')
export_reports()


## P1 — UA: entropy và teacher–student KL riêng biệt

**Theo yêu cầu tách thành phần của supervisor; bổ trợ lập luận cơ chế cho5EXK/QNjz:** `UA-noH` bỏ entropy, giữ KL; `UA-noD` bỏ KL, giữ entropy. So với F/A@25k, giữ UWRG, prototype auxiliary và phần còn lại. Hai nhánh chỉ chạy Stage A, không thêm TC gây nhiễu. Chi phí thêm50k/protocol.

**Đọc kết quả:** đối chiếu endpoint25k, score-quality theo iteration và từng class. Nếu hai thành phần bổ trợ, full UA có thể cho ranking/segmentation tốt hơn; nếu không, kết luận theo chỉ số thực đo. Để tách cả interaction có thêm `UA-neither` ở nhóm P2. Không so trực tiếp endpoint25k này với F/B@30k như một ablation cùng budget.


In [ ]:
run_arms('UA-noH', 'UA-noD')
export_reports()


## P1 — UWRG: region aggregation và bỏ gate

**Vc4u:** độ tin cậy underwater hay segment-median tạo gain? `G-uw-pixel` dùng cùng UW probe nhưng score từng pixel; `G-none` bỏ gate (factor1) trong cả hai stage. F là UW+median. Chi phí thêm60k/protocol.

**Đọc kết quả:** region so với pixel cần cùng probe; UW so với generic cần cùng aggregation. Với `G-generic-med` đã chạy, ba nhánh này giúp giải thích cơ chế mà không gom mọi khác biệt vào underwater prior. Nếu cần factorial đầy đủ, chạy `G-generic-pixel` ở P2. `G-confidence` không đồng nghĩa no-gate: identity probing vẫn giữ confidence weighting.


In [ ]:
run_arms('G-uw-pixel', 'G-none')
export_reports()


## P2 — Frozen reliability: bằng chứng bổ trợ ít tốn training

**RgDe/Vc4u/5EXK:** so UW/generic trên cùng model cố định, cùng ảnh và cùng magnitude calibration; xuất AURC, precision@75%, median/mean/pixel cùng các appearance strata đã định trước. Mặc định `DO_FROZEN=False` để ưu tiên dynamics và causal ablations; bật nếu còn thời gian. Không cần optimizer updates nhưng vẫn tốn inference và không có cơ chế checkpoint-safe time cap như training. Cell đo cả source-only và F/A@25k nếu anchor25k đã được commit. Kết quả UA25k có output riêng, dùng student/teacher và statistics thực từ checkpoint; nếu chưa có anchor thì ghi rõ chưa đo.

**Đọc kết quả:** AURC thấp hơn là ranking tốt hơn; khoảng tin cậy theo ảnh, nhiều draws được gộp trong ảnh. Không chuyển kết quả source-only thành kết luận gain adaptation. Cả hai dấu của contrast được giữ lại.


In [ ]:
if DO_FROZEN and remaining_hours() > 1.0:
    run_script('experiments/rebuttal/analyze.py', '--protocol', PROTOCOL, '--data', DATA, '--output', OUTPUT)
    from util import verified_pointer
    protocols = ('uiis_to_suim', 'usis10k_to_suim') if PROTOCOL == 'all' else (PROTOCOL,)
    for protocol in protocols:
        folder = OUTPUT/'full'/protocol/f'seed{SEED}'/'F/A'
        if (folder/'snapshots/025000.json').exists() and remaining_hours() > 1.0:
            _, checkpoint, _ = verified_pointer(folder, 'snapshots/025000.json')
            run_script('experiments/rebuttal/analyze.py', '--protocol', protocol, '--data', DATA,
                       '--output', OUTPUT/'frozen_ua25k', '--checkpoint', checkpoint, '--base', 'ua')
        else:
            print(protocol, ': UA25k frozen chưa đo (thiếu anchor hoặc ngân sách).')
else:
    print('Bỏ qua frozen analysis theo cấu hình/ngân sách; không ảnh hưởng các run training.')


## P2 — Độ nhạy split và các đối chiếu còn lại (tùy chọn)

**QNjz:** `S-20k` lấy F/A@20k và chạy10k TC; `S-27.5k` lấy UA-continuous@27.5k và chạy2.5k TC. Cả hai tổng30k, cùng warmup tuyệt đối và LR horizon đã khai báo. Thêm12.5k/protocol khi đã có các dependencies; đây là kiểm tra độ nhạy split, không chứng minh25k tối ưu toàn cục.

Các arm bổ sung: `UA-neither`25k, `G-generic-pixel`, `G-uw-mean`, `G-confidence`, `G-no-floor` mỗi30k/protocol. Chỉ bật `RUN_OPTIONAL` khi đã đủ reference/control chính; mọi kết quả được giữ. Có thể gọi riêng `run_arms('S-20k')` hoặc arm khác. Không quyết định nhánh nào được báo cáo theo hướng gain.


In [ ]:
if RUN_OPTIONAL:
    run_arms('S-20k', 'S-27.5k', 'UA-neither', 'G-generic-pixel', 'G-uw-mean', 'G-confidence', 'G-no-floor')
else:
    print('P2 chưa chạy. Đặt RUN_OPTIONAL=True hoặc gọi từng arm khi cần.')
export_reports()


## Báo cáo cuối / sau mỗi phiên

`results.csv` có endpoint nếu đã xong, hoặc last measured nếu chưa xong, cùng peak được chọn theo mIoU8 trên110 ảnh TEST. Tất cả per-class/secondary metrics ở cùng checkpoint. Mask TEST dùng evaluation và chọn peak; mask train bị loại khỏi adaptation. Cost có training/evaluation/checkpoint/candidate time và GPU memory; cộng shared Stage-A prefix đúng một lần khi so toàn pipeline. Không dùng tốc độ khác GPU hoặc diagnostic smoke để claim efficiency.

Đọc `SUPERVISOR_REPORT.md` và plot dynamics cùng `mismatch_5k_windows.csv`, `quality_over_time.csv`, `supervision_mass.csv`. Session pause không phải convergence/early stopping; report một phần không phải bảng kết quả hoàn chỉnh. Giữ toàn bộ folder Drive, code pin và full-state/banks cho phiên sau. Notebook không tự đổi batch/AMP khi OOM. Các thí nghiệm mới này bổ sung evidence, không được gắn thành run đã tạo headline scores trong manuscript.


In [ ]:
export_reports()
print('Drive output:', OUTPUT)
print('Code pin:', (OUTPUT/'CODE_REVISION.txt').read_text().strip())
print('Thời gian phiên đã dùng (giờ):', round((time.monotonic()-SESSION_STARTED)/3600, 2))
